In [ ]:
import os
import sys
from pathlib import Path
import pickle

# Resolve repo root whether notebook runs from repo root or notebooks/
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import torch
import lightning as L
import xarray as xr
import xskillscore as xs
from tqdm import tqdm
from pysteps.utils.spectral import rapsd

from src.models.unet_module import UnetLitModule
from src.models.gan_module import UnetGANLitModule
from src.models.ae_module import AutoencoderKL, EncoderLRES
from src.models.ldm_module import LatentDiffusion
from src.models.components.unet import DownscalingUnet
from src.models.components.ae import SimpleConvDecoder, SimpleConvEncoder
from src.models.components.ldm.denoiser import UNetModel, DDIMSampler
from src.models.components.ldm.conditioner import AFNOConditionerNetCascade
from src.data.downscaling_datamodule import DownscalingDataModule
from src.data.components.downscaling_dataset import DownscalingDataset

from utils.inference_utils import get_model_output
from utils.plotting_utils import (
    get_target_grid,
    from_torchtensor_to_xarray,
    quadratic_regrid_era5_xr_to_high_res,
    linear_regrid_era5_xr_to_high_res,
    show_metrics,
    show_spatial_errors,
    show_freq_distrib,
    show_power_spectra,
)

seed = 42
L.seed_everything(seed, workers=True)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Paths and experiment configuration
repo_root = Path.cwd()
trained_models_path = repo_root / "pretrained_models"
data_path = repo_root / "LDM-downscaling"
output_path = repo_root / "outputs" / "latest_inference"
output_path.mkdir(parents=True, exist_ok=True)

target_var = "2mT"  # "2mT" or "UV"
metadata_file_name = "metadata_test_paper_sample.csv"
denoising_steps = 100
pde_partial_ckpt_name = "pde_loss_model_checkpoint.ckpt"

static_vars = {
    "dtm_tif_file": str(data_path / "static_var" / "dtm_2km_domain_trim_EPSG3035.tif"),
    "lc_tif_file": str(data_path / "static_var" / "land_cover_classes_2km_domain_trim_EPSG3035.tif"),
    "lat_tif_file": str(data_path / "static_var" / "lat_2km_domain_trim_EPSG3035.tif"),
}
borders_file = str(data_path / "plotting_resources" / "borders_downscaling_domain_3035.geojson")

with open(data_path / "normalization_data.pkl", "rb") as f:
    norm_values = pickle.load(f)

target_channels = 1 if target_var == "2mT" else 2
target_vars = {"high_res": ["2mT"]} if target_var == "2mT" else {"high_res": ["U10", "V10"]}
target_vars["low_res"] = ["2mT", "PMSL", "U10", "V10", "dp2mT", "SST", "SNDPT", "TP", "SSRadIn", "Q850", "T850", "U850", "V850", "W850"]
tv_idxs = {tv: target_vars["low_res"].index(tv) for tv in target_vars["high_res"]}

print(f"target_var={target_var}, output={output_path}")

In [ ]:
def make_test_loader(nn_lowres: bool):
    dm = DownscalingDataModule(
        data_dir=str(data_path),
        target_vars=target_vars,
        batch_size=1,
        num_workers=1,
    )
    dm.data_test = DownscalingDataset(
        str(data_path),
        target_vars=target_vars,
        nn_lowres=nn_lowres,
        static_vars=static_vars,
        metadata_file_name=metadata_file_name,
    )
    return dm.test_dataloader(), len(dm.data_test)

def denorm_field(model_key: str, variable: str, x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    src = "CMCC" if model_key != "ERA5" else "ERA5"
    return x * norm_values[src]["std"][variable] + norm_values[src]["mean"][variable]

def compute_ws(sample_dict):
    if "U10" in sample_dict and "V10" in sample_dict:
        sample_dict["WS"] = np.sqrt(sample_dict["U10"] ** 2 + sample_dict["V10"] ** 2)

def add_records(records, model_name, ts_ns, sample_dict):
    for var_name, field in sample_dict.items():
        records.append(
            {
                "input_var": "all",
                "target_var": target_var,
                "model": model_name,
                "variable": var_name,
                "spat_distr": np.asarray(field, dtype=np.float32),
                "min": float(np.nanpercentile(field, 0.5)),
                "max": float(np.nanpercentile(field, 99.5)),
                "time_step": ts_ns,
            }
        )

In [ ]:
# Load models
loss = torch.nn.Identity()
in_ch = 18 + 14
out_ch = target_channels

unet_model = UnetLitModule.load_from_checkpoint(
    str(trained_models_path / f"UNET_{target_var}.ckpt"),
    net=DownscalingUnet(in_ch=in_ch, out_ch=out_ch),
    loss=loss,
    ckpt_path=None,
    strict=False,
).eval().to(device)

gan_model = UnetGANLitModule.load_from_checkpoint(
    str(trained_models_path / f"GAN_{target_var}.ckpt"),
    net=DownscalingUnet(in_ch=in_ch, out_ch=out_ch),
    loss=loss,
    ckpt_path=None,
    strict=False,
).eval().to(device)

def build_ldm_model():
    return LatentDiffusion(
        denoiser=UNetModel(
            in_channels=32 * target_channels,
            model_channels=256,
            out_channels=32 * target_channels,
            num_res_blocks=2,
            attention_resolutions=[1, 2],
            dims=2,
            channel_mult=[1, 2, 4],
            num_heads=8,
            context_ch=[256, 512, 1024],
        ),
        autoencoder=AutoencoderKL(
            encoder=SimpleConvEncoder(in_dim=target_channels, levels=3),
            decoder=SimpleConvDecoder(in_dim=target_channels, levels=3),
            ae_flag="residual",
            unet_regr=unet_model,
        ),
        context_encoder=AFNOConditionerNetCascade(
            autoencoder=[AutoencoderKL(encoder=SimpleConvEncoder(in_dim=18, levels=3, ch_mult=3), decoder=None), EncoderLRES()],
            train_autoenc=False,
            cascade_depth=3,
            embed_dim=[128, 24],
            analysis_depth=[4, 4],
            afno_fusion=True,
            input_size_ratios=[1, 1],
            embed_dim_out=256,
        ),
        ae_load_state_file=str(trained_models_path / f"VAE_residual_{target_var}.ckpt"),
        parameterization="v",
    )

base_ldm_ckpt = trained_models_path / f"LDM_residual_{target_var}.ckpt"

# Original LDM_res model
ldm_model = LatentDiffusion.load_from_checkpoint(
    str(base_ldm_ckpt),
    denoiser=UNetModel(
        in_channels=32 * target_channels,
        model_channels=256,
        out_channels=32 * target_channels,
        num_res_blocks=2,
        attention_resolutions=[1, 2],
        dims=2,
        channel_mult=[1, 2, 4],
        num_heads=8,
        context_ch=[256, 512, 1024],
    ),
    autoencoder=AutoencoderKL(
        encoder=SimpleConvEncoder(in_dim=target_channels, levels=3),
        decoder=SimpleConvDecoder(in_dim=target_channels, levels=3),
        ae_flag="residual",
        unet_regr=unet_model,
    ),
    context_encoder=AFNOConditionerNetCascade(
        autoencoder=[AutoencoderKL(encoder=SimpleConvEncoder(in_dim=18, levels=3, ch_mult=3), decoder=None), EncoderLRES()],
        train_autoenc=False,
        cascade_depth=3,
        embed_dim=[128, 24],
        analysis_depth=[4, 4],
        afno_fusion=True,
        input_size_ratios=[1, 1],
        embed_dim_out=256,
    ),
    ae_load_state_file=str(trained_models_path / f"VAE_residual_{target_var}.ckpt"),
    parameterization="v",
).eval().to(device)
ldm_sampler = DDIMSampler(ldm_model)

# PDE-merged LDM model (same setup as models_inference.ipynb)
ldm_pde_model = build_ldm_model().to(device)
full_ckpt = torch.load(base_ldm_ckpt, map_location=device, weights_only=False)
partial_ckpt = torch.load(trained_models_path / pde_partial_ckpt_name, map_location=device, weights_only=False)
combined_state = full_ckpt["state_dict"].copy()
combined_state.update(partial_ckpt["state_dict"])
ldm_pde_model.load_state_dict(combined_state, strict=False)
ldm_pde_model.eval()
ldm_pde_sampler = DDIMSampler(ldm_pde_model)

In [ ]:
# Run fresh inference and build compact results table (no per-timestep image dumps)
records = []

unet_loader, n_test_unet = make_test_loader(nn_lowres=True)
ldm_loader, n_test_ldm = make_test_loader(nn_lowres=False)
n_test = min(n_test_unet, n_test_ldm)

unet_iter = iter(unet_loader)
gan_iter = iter(unet_loader)
base_iter = iter(unet_loader)
ldm_iter = iter(ldm_loader)

target_grid_low = get_target_grid("low")
target_grid_high = get_target_grid("high")

for _ in tqdm(range(n_test), desc="Inference"):
    # UNET/GAN/COSMO/ERA5 share nn_lowres=True loader format
    el_base = next(base_iter)
    low_res, high_res, ts_base = el_base
    ts_base = pd.to_datetime(int(ts_base))

    era5_sample = {}
    cosmo_sample = {}
    for tv in target_vars["high_res"]:
        era5_sample[tv] = denorm_field("ERA5", tv, low_res[0, tv_idxs[tv], :, :])
        cosmo_sample[tv] = denorm_field("COSMO-CLM", tv, high_res[0, target_vars['high_res'].index(tv), :, :])
    compute_ws(era5_sample)
    compute_ws(cosmo_sample)
    add_records(records, "ERA5", ts_base, era5_sample)
    add_records(records, "COSMO-CLM", ts_base, cosmo_sample)

    quadratic_sample = {}
    for tv in target_vars["high_res"]:
        era5_xr = from_torchtensor_to_xarray(low_res[0, tv_idxs[tv], :, :], target_grid_low, coords_name="y_x")
        interp_q = quadratic_regrid_era5_xr_to_high_res(era5_xr, target_grid_high)
        quadratic_sample[tv] = denorm_field("ERA5", tv, interp_q.values)
    compute_ws(quadratic_sample)
    add_records(records, "Quadratic Interp.", ts_base, quadratic_sample)

    linear_sample = {}
    for tv in target_vars["high_res"]:
        era5_xr = from_torchtensor_to_xarray(low_res[0, tv_idxs[tv], :, :], target_grid_low, coords_name="y_x")
        interp_l = linear_regrid_era5_xr_to_high_res(era5_xr, target_grid_high)
        linear_sample[tv] = denorm_field("ERA5", tv, interp_l.values)
    compute_ws(linear_sample)
    add_records(records, "Linear Interp.", ts_base, linear_sample)

    el_unet = next(unet_iter)
    pred_unet, ts_unet = get_model_output("unet-like", unet_model, el_unet)
    ts_unet = pd.to_datetime(int(ts_unet))
    unet_sample = {tv: denorm_field("UNET", tv, pred_unet[0, i, :, :]) for i, tv in enumerate(target_vars["high_res"])}
    compute_ws(unet_sample)
    add_records(records, "UNET", ts_unet, unet_sample)

    el_gan = next(gan_iter)
    pred_gan, ts_gan = get_model_output("unet-like", gan_model, el_gan)
    ts_gan = pd.to_datetime(int(ts_gan))
    gan_sample = {tv: denorm_field("GAN", tv, pred_gan[0, i, :, :]) for i, tv in enumerate(target_vars["high_res"])}
    compute_ws(gan_sample)
    add_records(records, "GAN", ts_gan, gan_sample)

    el_ldm = next(ldm_iter)

    pred_ldm, ts_ldm = get_model_output("ldm", ldm_model, el_ldm, sampler=ldm_sampler, num_diffusion_iters=denoising_steps)
    ts_ldm = pd.to_datetime(int(ts_ldm))
    ldm_sample = {tv: denorm_field("LDM_res", tv, pred_ldm[0, i, :, :]) for i, tv in enumerate(target_vars["high_res"])}
    compute_ws(ldm_sample)
    add_records(records, "LDM_res", ts_ldm, ldm_sample)

    pred_ldm_pde, ts_ldm_pde = get_model_output("ldm", ldm_pde_model, el_ldm, sampler=ldm_pde_sampler, num_diffusion_iters=denoising_steps)
    ts_ldm_pde = pd.to_datetime(int(ts_ldm_pde))
    ldm_pde_sample = {tv: denorm_field("LDM_PDE_res", tv, pred_ldm_pde[0, i, :, :]) for i, tv in enumerate(target_vars["high_res"])}
    compute_ws(ldm_pde_sample)
    add_records(records, "LDM_PDE_res", ts_ldm_pde, ldm_pde_sample)

results_df = pd.DataFrame(records).sort_values(["time_step", "model", "variable"]).reset_index(drop=True)
results_file = output_path / f"results_latest_{target_var}.pkl"
results_df.to_pickle(results_file)
print(f"Saved: {results_file} (rows={len(results_df)})")

expected_models = ["ERA5", "COSMO-CLM", "Quadratic Interp.", "Linear Interp.", "UNET", "GAN", "LDM_res", "LDM_PDE_res"]
missing_models = [m for m in expected_models if m not in results_df["model"].unique()]
if missing_models:
    raise RuntimeError(f"Missing model outputs: {missing_models}")
print("Models in results:", sorted(results_df["model"].unique().tolist()))

In [ ]:
# Metrics (RMSE, R2, BIAS, PCC) against COSMO-CLM
target_grid_high_res = get_target_grid("high")
metrics_rows = []
model_names = sorted(results_df["model"].unique())
metric_models = [m for m in model_names if m not in ["ERA5", "COSMO-CLM"]]

for model_name in metric_models:
    model_rows = results_df[results_df["model"] == model_name]
    for _, row in model_rows.iterrows():
        ref_row = results_df[
            (results_df["model"] == "COSMO-CLM")
            & (results_df["time_step"] == row["time_step"])
            & (results_df["variable"] == row["variable"])
        ]
        if ref_row.empty:
            continue
        ref = ref_row.iloc[0]["spat_distr"]
        pred = row["spat_distr"]
        ref_xr = from_torchtensor_to_xarray(torch.as_tensor(ref), target_grid_high_res, coords_name="y_x")
        pred_xr = from_torchtensor_to_xarray(torch.as_tensor(pred), target_grid_high_res, coords_name="y_x")
        vals = {
            "RMSE": xs.rmse(pred_xr, ref_xr).item(),
            "R2": xs.r2(pred_xr, ref_xr).item(),
            "BIAS": xs.me(pred_xr, ref_xr).item(),
            "PCC": xs.pearson_r(pred_xr, ref_xr).item(),
        }
        for metric_name, metric_value in vals.items():
            metrics_rows.append({
                "model": model_name,
                "target_var": row["target_var"],
                "var": row["variable"],
                "metric": metric_name,
                "value": metric_value,
            })

metrics_df = pd.DataFrame(metrics_rows)
metrics_file = output_path / f"metrics_latest_{target_var}.pkl"
metrics_df.to_pickle(metrics_file)
print(f"Saved: {metrics_file} (rows={len(metrics_df)})")
show_metrics(metrics_df, str(output_path) + "/")

In [ ]:
# Spatial error map (time-mean prediction minus COSMO-CLM)
spatial_rows = []
plot_variables = ["2mT"] if target_var == "2mT" else ["WS"]

for var_name in plot_variables:
    ref_sub = results_df[(results_df["model"] == "COSMO-CLM") & (results_df["variable"] == var_name)]
    ref_by_ts = {r["time_step"]: r["spat_distr"] for _, r in ref_sub.iterrows()}
    for model_name in [m for m in model_names if m not in ["ERA5", "COSMO-CLM"]]:
        pred_sub = results_df[(results_df["model"] == model_name) & (results_df["variable"] == var_name)]
        errs = []
        for _, row in pred_sub.iterrows():
            if row["time_step"] in ref_by_ts:
                errs.append(np.asarray(row["spat_distr"]) - np.asarray(ref_by_ts[row["time_step"]]))
        if len(errs) == 0:
            continue
        mean_err = np.nanmean(np.stack(errs, axis=0), axis=0)
        spatial_rows.append({
            "variable": var_name,
            "model": model_name,
            "spat_distr": mean_err,
            "min": float(np.nanpercentile(mean_err, 0.5)),
            "max": float(np.nanpercentile(mean_err, 99.5)),
        })

spat_err_df = pd.DataFrame(spatial_rows)
spat_err_file = output_path / f"spatial_errors_latest_{target_var}.pkl"
spat_err_df.to_pickle(spat_err_file)
print(f"Saved: {spat_err_file} (rows={len(spat_err_df)})")
show_spatial_errors(spat_err_df, target_res="high", output_dir=str(output_path) + "/", borders_file=borders_file)

In [ ]:
# Frequency distribution and power spectra
freq_rows = []
spectra_rows = []
plot_variables = ["2mT"] if target_var == "2mT" else ["WS"]
bins = 180

for var_name in plot_variables:
    var_sub = results_df[results_df["variable"] == var_name]
    vmin = np.nanpercentile(np.concatenate([np.ravel(x) for x in var_sub["spat_distr"].values]), 0.1)
    vmax = np.nanpercentile(np.concatenate([np.ravel(x) for x in var_sub["spat_distr"].values]), 99.9)
    edges = np.linspace(vmin, vmax, bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    for model_name in model_names:
        model_sub = var_sub[var_sub["model"] == model_name]
        if model_sub.empty:
            continue
        flat = np.concatenate([np.ravel(x) for x in model_sub["spat_distr"].values])
        hist, _ = np.histogram(flat, bins=edges)
        hist = hist.astype(np.float64) + 1e-8
        freq_rows.append({"model": model_name, "variable": var_name, "x_s": centers, "freq_distr": hist})

        spec_list = []
        fft_freq = None
        for field in model_sub["spat_distr"].values:
            spec, freq = rapsd(np.asarray(field), return_freq=True)
            spec_list.append(spec)
            if fft_freq is None:
                fft_freq = freq
        spectra_rows.append({
            "model": model_name,
            "variable": var_name,
            "spectra": np.stack(spec_list, axis=0),
            "fft_freq": fft_freq,
        })

freq_df = pd.DataFrame(freq_rows)
spectra_df = pd.DataFrame(spectra_rows)

freq_file = output_path / f"freq_distribution_latest_{target_var}.pkl"
spectra_file = output_path / f"power_spectra_latest_{target_var}.pkl"
freq_df.to_pickle(freq_file)
spectra_df.to_pickle(spectra_file)

print(f"Saved: {freq_file} (rows={len(freq_df)})")
print(f"Saved: {spectra_file} (rows={len(spectra_df)})")
show_freq_distrib(freq_df, output_dir=str(output_path) + "/")
show_power_spectra(spectra_df, output_dir=str(output_path) + "/")